In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

In [ ]:
# Load the data
all_workers = pd.read_csv("output/workers_merged.csv")

In [ ]:
# Filter on residence and work location
workers_in_brussels = all_workers[all_workers["works_in_brussels"]].copy()
workers_outside_brussels = all_workers[~all_workers["works_in_brussels"]].copy()

commuters = all_workers[~all_workers["lives_in_brussels"]].copy()
residents = all_workers[all_workers["lives_in_brussels"]].copy()

# TOTALS

In [ ]:
print(f"Total number of workers: {len(all_workers)}")

print(f"Number of residents: {len(residents)}")
print(f"Number of commuters: {len(commuters)}")

print(f"Number of people working in Brussels: {len(workers_in_brussels)}")
print(f"Number of people working outside of Brussels: {len(workers_outside_brussels)}")

# MODE SPLIT

In [ ]:
# Split of transport mode
print("All")
mode_split = (
    all_workers["assigned_mode"]
    .value_counts(dropna=False)
    .rename_axis("mode")
    .to_frame("count")
)

mode_split["share_pct"] = (mode_split["count"] / mode_split["count"].sum() * 100).round(2)
print(mode_split)

# Only people who work in Brussels
print("\nWork in Brussels")
mode_split = (
    workers_in_brussels["assigned_mode"]
    .value_counts(dropna=False)
    .rename_axis("mode")
    .to_frame("count")
)

mode_split["share_pct"] = (mode_split["count"] / mode_split["count"].sum() * 100).round(2)
print(mode_split)

# Only Brussels residents
print("\nResidents")
mode_split = (
    residents["assigned_mode"]
    .value_counts(dropna=False)
    .rename_axis("mode")
    .to_frame("count")
)

mode_split["share_pct"] = (mode_split["count"] / mode_split["count"].sum() * 100).round(2)
print(mode_split)

# Only Brussels commuters
print("\nCommuters")
mode_split = (
    commuters["assigned_mode"]
    .value_counts(dropna=False)
    .rename_axis("mode")
    .to_frame("count")
)

mode_split["share_pct"] = (mode_split["count"] / mode_split["count"].sum() * 100).round(2)
print(mode_split)

In [ ]:
# Modal split by commute distance bands
distance_bins = [0, 2, 5, 10, 20, 40, np.inf]
distance_labels = ["0-2", "2-5", "5-10", "10-20", "20-40", "40+"]

tmp = all_workers[["commute_dist_km", "assigned_mode"]].dropna().copy()
tmp["distance_band"] = pd.cut(
    tmp["commute_dist_km"],
    bins=distance_bins,
    labels=distance_labels,
    right=False,
)

# Absolute counts by distance band and mode
mode_by_distance_counts = pd.crosstab(tmp["distance_band"], tmp["assigned_mode"])

# Percent shares within each distance band
mode_by_distance_share = (
    pd.crosstab(tmp["distance_band"], tmp["assigned_mode"], normalize="index") * 100
).round(2)

print("Counts:")
print(mode_by_distance_counts)

print("\nShare (%) within each distance band:")
print(mode_by_distance_share)

# Optional quick visualization
mode_by_distance_share.plot(kind="bar", stacked=True, figsize=(10, 5))

# CHECK DEPARTURE TIMES

In [ ]:
agents_missing_departure_from_work = all_workers[all_workers["departure_from_work"].isna()].copy()

print(f"Agents with missing departure_from_work: {len(agents_missing_departure_from_work)}")
agents_missing_departure_from_work.head()

In [ ]:
agents_missing_work_location = all_workers[all_workers["work_y"].isna()].copy()

print(f"Agents with missing work location: {len(agents_missing_work_location)}")
agents_missing_work_location.head()

# PROFESSION TYPE

In [ ]:
# Stats for professional_status in all_workers
professional_status_counts = all_workers["professional_status"].value_counts(dropna=False)
professional_status_percent = all_workers["professional_status"].value_counts(normalize=True, dropna=False).mul(100).round(1)

professional_status_stats = (
    pd.DataFrame({
        "count": professional_status_counts,
        "percent": professional_status_percent,
    })
    .reset_index()
    .rename(columns={"index": "professional_status"})
)

print(f"Total agents in all_workers: {len(all_workers):,}")
professional_status_stats

# DEPARTURE TIMES

In [ ]:
all_workers['departure_home_to_work'].describe()


In [ ]:
x = all_workers["departure_home_to_work"].dropna()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(x, bins=48, color="steelblue", edgecolor="black")

# convert minute-based x-axis to hours
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, pos: f"{v/60:.0f}"))
ax.set_xlabel("Departure time (hours after midnight)")
ax.set_ylabel("Count")
ax.set_title("Histogram of departure_home_to_work")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
x = all_workers["departure_from_work"].dropna()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(x, bins=48, color="steelblue", edgecolor="black")

# convert minute-based x-axis to hours
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, pos: f"{v/60:.0f}"))
ax.set_xlabel("Departure time (hours after midnight)")
ax.set_ylabel("Count")
ax.set_title("Histogram of departure_from_work")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


# COMMUTING DISTANCE

## Residents

In [ ]:
residents["commute_dist_km"].describe()

In [ ]:
distance_bins_residents = [0, 1, 2, 5, 10, 15, 25, 40, np.inf]
distance_labels_residents = ["0-1", "1-2", "2-5", "5-10", "10-15", "15-25", "25-40", "40+"]

tmp = residents[["commute_dist_km"]].dropna().copy()
tmp["distance_band"] = pd.cut(
    tmp["commute_dist_km"],
    bins=distance_bins_residents,
    labels=distance_labels_residents,
    right=False,
    include_lowest=True,
)

# Keep the bins in the exact order defined above and convert to percentages
band_counts = tmp["distance_band"].value_counts().reindex(distance_labels_residents, fill_value=0)
band_pct = (band_counts / band_counts.sum() * 100).round(2)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(distance_labels_residents, band_pct.values, color="steelblue", edgecolor="black")

ax.set_xlabel("Commuting distance in km")
ax.set_ylabel("Percentage (%)")
ax.set_title("Histogram of commuting distance (beeline distance * 1.3)")
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, max(band_pct.max() * 1.15, 1))

plt.tight_layout()
plt.show()

In [ ]:
residents_before_teleworking = pd.read_csv('../3_work_assignment/output/workers_with_work_addresses.csv')

In [ ]:
DETOUR_FACTOR = 1.3

residents_before_teleworking["commute_dist_km"] = (
    np.sqrt(
        (residents_before_teleworking["home_x"] - residents_before_teleworking["work_x"])**2 +
        (residents_before_teleworking["home_y"] - residents_before_teleworking["work_y"])**2
    ) / 1000 * DETOUR_FACTOR
)

In [ ]:
distance_bins_residents = [0, 0.5, 1, 2, 5, 10, 15, 25, 40, np.inf]
distance_labels_residents = ["0-0.5", "0.5-1", "1-2", "2-5", "5-10", "10-15", "15-25", "25-40", "40+"]

tmp = residents_before_teleworking[["commute_dist_km"]].dropna().copy()
tmp["distance_band"] = pd.cut(
    tmp["commute_dist_km"],
    bins=distance_bins_residents,
    labels=distance_labels_residents,
    right=False,
    include_lowest=True,
)

# Keep the bins in the exact order defined above and convert to percentages
band_counts = tmp["distance_band"].value_counts().reindex(distance_labels_residents, fill_value=0)
band_pct = (band_counts / band_counts.sum() * 100).round(2)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(distance_labels_residents, band_pct.values, color="steelblue", edgecolor="black")

ax.set_xlabel("Commuting distance in km")
ax.set_ylabel("Percentage (%)")
ax.set_title("Histogram of commuting distance (beeline distance * 1.3)")
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, max(band_pct.max() * 1.15, 1))

plt.tight_layout()
plt.show()

<!-- commute_dist_km -->

## All working in Brussels 

In [ ]:
workers_in_brussels["commute_dist_km"].describe()

In [ ]:
distance_bins_residents = [0, 1, 2, 5, 10, 15, 25, 40, np.inf]
distance_labels_residents = ["0-1", "1-2", "2-5", "5-10", "10-15", "15-25", "25-40", "40+"]

tmp = workers_in_brussels[["commute_dist_km"]].dropna().copy()
tmp["distance_band"] = pd.cut(
    tmp["commute_dist_km"],
    bins=distance_bins_residents,
    labels=distance_labels_residents,
    right=False,
    include_lowest=True,
)

# Keep the bins in the exact order defined above and convert to percentages
band_counts = tmp["distance_band"].value_counts().reindex(distance_labels_residents, fill_value=0)
band_pct = (band_counts / band_counts.sum() * 100).round(2)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(distance_labels_residents, band_pct.values, color="steelblue", edgecolor="black")

ax.set_xlabel("Commuting distance in km")
ax.set_ylabel("Percentage (%)")
ax.set_title("Histogram of commuting distance (beeline distance * 1.3)")
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, max(band_pct.max() * 1.15, 1))

plt.tight_layout()
plt.show()

## Commuters

In [ ]:
commuters["commute_dist_km"].describe()

In [ ]:
distance_bins_residents = [0, 1, 2, 5, 10, 15, 25, 40, np.inf]
distance_labels_residents = ["0-1", "1-2", "2-5", "5-10", "10-15", "15-25", "25-40", "40+"]

tmp = commuters[["commute_dist_km"]].dropna().copy()
tmp["distance_band"] = pd.cut(
    tmp["commute_dist_km"],
    bins=distance_bins_residents,
    labels=distance_labels_residents,
    right=False,
    include_lowest=True,
)

# Keep the bins in the exact order defined above and convert to percentages
band_counts = tmp["distance_band"].value_counts().reindex(distance_labels_residents, fill_value=0)
band_pct = (band_counts / band_counts.sum() * 100).round(2)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(distance_labels_residents, band_pct.values, color="steelblue", edgecolor="black")

ax.set_xlabel("Commuting distance in km")
ax.set_ylabel("Percentage (%)")
ax.set_title("Histogram of commuting distance (beeline distance * 1.3)")
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, max(band_pct.max() * 1.15, 1))

plt.tight_layout()
plt.show()